In [ ]:
import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors


def knn_graph(points, k, mutual=False):

    n = len(points)

    nbrs = NearestNeighbors(n_neighbors=k+1)
    nbrs.fit(points)

    _, indices = nbrs.kneighbors(points)

    G = nx.Graph()
    G.add_nodes_from(range(n))
    if mutual:

        neighbor_sets = [set(row[1:]) for row in indices]

        for i in range(n):
            for j in indices[i][1:]:
                if i in neighbor_sets[j]:
                    G.add_edge(i,j)

    else:

        for i in range(n):
            for j in indices[i][1:]:
                G.add_edge(i,j)

    return G


from collections import defaultdict
def build_edges_triangles(G):
    # -------------------------
    # edges
    # -------------------------

    edges = []
    edge_id = {}

    for i,j in G.edges():

        if i>j:
            i,j=j,i

        edge_id[(i,j)] = len(edges)
        edges.append((i,j))

    # -------------------------
    # adjacency
    # -------------------------

    neighbors = {
        v:set(G.neighbors(v))
        for v in G.nodes()
    }

    # -------------------------
    # triangles
    # -------------------------

    triangles = []

    for i in G.nodes():

        Ni = neighbors[i]

        for j in Ni:

            if j<=i:
                continue

            common = Ni & neighbors[j]

            for k in common:

                if k<=j:
                    continue

                triangles.append((i,j,k))

    return edges, edge_id, triangles


from scipy.sparse import csr_matrix
def build_B1(n_vertices, edges):

    rows=[]
    cols=[]
    vals=[]

    for e,(i,j) in enumerate(edges):

        rows.extend([i,j])
        cols.extend([e,e])
        vals.extend([-1,1])

    return csr_matrix(
        (vals,(rows,cols)),
        shape=(n_vertices,len(edges))
    )
def build_B2(edges, edge_id, triangles):
    rows=[]
    cols=[]
    vals=[]

    for t,(i,j,k) in enumerate(triangles):

        e0=edge_id[(j,k)]
        e1=edge_id[(i,k)]
        e2=edge_id[(i,j)]

        rows.extend([e0,e1,e2])
        cols.extend([t,t,t])

        vals.extend([1,-1,1])

    return csr_matrix(
        (vals,(rows,cols)),
        shape=(len(edges),len(triangles))
    )
def edge_flow_from_vector_field(edges, points, vectors):
    omega=np.zeros(len(edges))

    for m,(i,j) in enumerate(edges):

        e=points[j]-points[i]

        L=np.linalg.norm(e)

        if L==0:
            continue

        t=e/L

        v=0.5*(vectors[i]+vectors[j])

        omega[m]=np.dot(v,t)

    return omega
from scipy.sparse.linalg import lsmr
def hodge_decomposition(B1, B2, omega):

    # vertex potential
    alpha = lsmr(B1.T, omega)[0]

    exact = B1.T @ alpha

    x = lsmr(B2, omega)[0]
    coexact = B2 @ x

    harmonic = omega - exact - coexact

    return alpha, exact, coexact, harmonic


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import transforms
from torchvision.datasets import STL10
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from timm.models.vision_transformer import VisionTransformer

weight_root_path="PATH"

class SimCLR(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = VisionTransformer(
            img_size=224,
            patch_size=16,
            in_chans=3,
            num_classes=0,     
            embed_dim=256,     
            depth=4,           
            num_heads=4,       
            mlp_ratio=4.0,
            qkv_bias=True,
        )

        feature_dim = self.encoder.num_features

        self.projector = nn.Sequential(
            nn.Linear(feature_dim,2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048,128)
        )

    def forward(self,x):

        h = self.encoder(x)
        z = self.projector(h)

        z = F.normalize(z,dim=1)

        return h,z
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import STL10
from torchvision import transforms
import numpy as np


# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ----------------------
# STL10 labeled dataset
# ----------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = STL10(
    root="./data",
    split="train",   # labeled train data
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4
)


In [ ]:
# ----------------------
# SimCLR model load
# ----------------------
full_feature_list=[]
full_projs_list=[]
model = SimCLR()
weightpaths=["PATHS FOR WEIGHTS HERE"
             ]
for weightpath in weightpaths:
    print(weightpath)
    checkpoint = torch.load(
        weightpath,
        map_location=device
    )
    print("checkpoint loaded")
    model.load_state_dict(checkpoint)

    model = model.to(device)
    model.eval()


    # ----------------------
    # Feature extraction
    # ----------------------

    tokens_list = []
    labels_list = []
    proj_list = []

    with torch.no_grad():

        for images, targets in loader:
            images = images.to(device)
            h,z = model(images)
            tokens_list.append(h.cpu())
            proj_list.append(z.cpu())
            labels_list.append(targets)

    tokens = torch.cat(tokens_list, dim=0)   
    projs = torch.cat(proj_list,dim=0)
    labels = torch.cat(labels_list, dim=0)
    print("token calculated")

    tokens = tokens.numpy()
    projs = projs.numpy()
    labels = labels.numpy()
    full_feature_list.append(tokens)
    full_projs_list.append(projs)

In [ ]:
full_feature_arr=np.array(full_feature_list)
print(full_feature_arr.shape)
move_vector=[full_feature_arr[i+1]-full_feature_arr[i] for i in range (0,len(full_feature_arr)-1)]
move_vector=np.array(move_vector)
move_vector_relative=move_vector-np.mean(move_vector,axis=1,keepdims=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve


from scipy.interpolate import griddata
import matplotlib.colors as mcolors

def prepare_graph_layout(edges, n_nodes, seed=0):

    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    G.add_edges_from(edges)
    pos = nx.spring_layout(G, seed=seed, iterations=30)
    return G, pos



def edge_to_potential(edges, edge_values, n_nodes):

    m = len(edges)

    row = []
    col = []
    data = []

    for k, (i, j) in enumerate(edges):

        row.extend([k, k])
        col.extend([i, j])
        data.extend([-1, 1])

    B = coo_matrix(
        (data, (row, col)),
        shape=(m, n_nodes)
    )

    phi = lsqr(B, edge_values)[0]

    phi -= phi.mean()

    return phi

def draw_potential(
        pos,
        labels,
        potential,
        title="Potential",
        node_size=3
):

    fig, ax = plt.subplots(figsize=(10,10))

    xy = np.array([
        pos[i]
        for i in range(len(labels))
    ])

    sc = ax.scatter(
        xy[:,0],
        xy[:,1],
        c=potential,
        cmap="rainbow",
        s=node_size*3,
    )

    plt.colorbar(sc, ax=ax)

    ax.set_title(title)

    ax.axis("equal")
    ax.axis("off")

    plt.show()

def edge_flow_to_node_vector(
        edges,
        values,
        pos,
        n_nodes
):

    vectors = np.zeros((n_nodes,2))
    count = np.zeros(n_nodes)

    for (i,j),f in zip(edges,values):

        p0 = np.array(pos[i])
        p1 = np.array(pos[j])

        d = p1-p0

        L = np.linalg.norm(d)

        if L==0:
            continue

        d /= L

        vectors[i] += f*d
        vectors[j] += f*d

        count[i]+=1
        count[j]+=1

    count[count==0]=1

    vectors /= count[:,None]

    return vectors


def smooth_node_vectors(
        edges,
        vectors,
        n_nodes,
        smooth=2.0
):

    row=[]
    col=[]
    data=[]

    degree=np.zeros(n_nodes)

    for i,j in edges:

        row.extend([i,j])
        col.extend([j,i])
        data.extend([1,1])

        degree[i]+=1
        degree[j]+=1

    A=csr_matrix(
        (data,(row,col)),
        shape=(n_nodes,n_nodes)
    )

    D=csr_matrix(
        (
            degree,
            (
                np.arange(n_nodes),
                np.arange(n_nodes)
            )
        ),
        shape=(n_nodes,n_nodes)
    )

    L=D-A

    M=csr_matrix(np.eye(n_nodes))+smooth*L

    vx=spsolve(M,vectors[:,0])
    vy=spsolve(M,vectors[:,1])

    return np.column_stack((vx,vy))


def draw_potential_ax(
    ax,
    pos,
    labels,
    potential,
    title="Potential",
    node_size=3
):

    xy = np.array([pos[i] for i in range(len(labels))])

    sc = ax.scatter(
        xy[:,0],
        xy[:,1],
        c=potential,
        cmap="rainbow",
        s=node_size*3,
    )

    plt.colorbar(sc, ax=ax)

    ax.set_title(title)
    ax.axis("equal")
    ax.axis("off")

def draw_streamline_ax(
    ax,
    pos,
    labels,
    vectors,
    title,
    grid_size=200,
    density=1.5,
    maxlength=8,
    node_size=10
):

    xy=np.array([pos[i] for i in range(len(labels))])

    x=xy[:,0]
    y=xy[:,1]

    xi=np.linspace(x.min(),x.max(),grid_size)
    yi=np.linspace(y.min(),y.max(),grid_size)

    X,Y=np.meshgrid(xi,yi)

    U=griddata(
        (x,y),
        vectors[:,0],
        (X,Y),
        method="cubic",
        fill_value=0
    )

    V=griddata(
        (x,y),
        vectors[:,1],
        (X,Y),
        method="cubic",
        fill_value=0
    )

    speed=np.sqrt(U**2+V**2)
    vmin = np.percentile(speed, 0)
    vmax = np.percentile(speed, 60)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    strm=ax.streamplot(
        X,
        Y,
        U,
        V,
        density=density,
        color=speed,
        cmap="Purples",
        linewidth=1.5,
        arrowsize=1.2,
        maxlength=maxlength,
        norm=norm
    )
    plt.colorbar(strm.lines, ax=ax, label="Speed")

    ax.scatter(
        x,
        y,
        c=labels,
        vmin=0,
        vmax=9,###Fixed Just for Visualization
        cmap="tab10",
        s=node_size,
        alpha=0.5,
        zorder=5
    )

    ax.set_title(title)
    ax.axis("equal")
    ax.axis("off")


### potentials are negated for intuitive visualizations

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

def process_one(index):

    points = full_feature_arr[index]
    vectors = move_vector_relative[index]

    #################################################
    # 1. kNN graph
    #################################################

    G = knn_graph(points, k=100)

    #################################################
    # 2.
    #################################################

    edges, edge_id, triangles = build_edges_triangles(G)

    #################################################
    # 3.
    #################################################

    B1 = build_B1(len(points), edges)
    B2 = build_B2(edges, edge_id, triangles)

    #################################################
    # 4.
    #################################################

    omega = edge_flow_from_vector_field(
        edges,
        points,
        vectors,
    )

    #################################################
    # 5.
    #################################################

    alpha, exact, coexact, harmonic = hodge_decomposition(
        B1,
        B2,
        omega,
    )

    #################################################
    # layout
    #################################################

    G_layout, pos = prepare_graph_layout(
        edges,
        n_nodes=len(labels),
    )

    #################################################
    # Figure
    #################################################

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(18,18)
    )

    #################################################
    # Exact Potential
    #################################################

    potential = -edge_to_potential(
        edges,
        exact,
        len(labels),
    )

    draw_potential_ax(
        axes[0,0],
        pos,
        labels,
        potential,
        "Exact Potential",
    )

    #################################################
    # Exact Stream
    #################################################

    vec = edge_flow_to_node_vector(
        edges,
        exact,
        pos,
        len(labels),
    )

    vec = smooth_node_vectors(
        edges,
        vec,
        len(labels),
        smooth=0.05,
    )

    draw_streamline_ax(
        axes[0,1],
        pos,
        labels,
        vec,
        "Exact Streamline",
        density=1.5,
        maxlength=10,
    )

    #################################################
    # Coexact
    #################################################

    vec = edge_flow_to_node_vector(
        edges,
        coexact,
        pos,
        len(labels),
    )

    vec = smooth_node_vectors(
        edges,
        vec,
        len(labels),
        smooth=0.05,
    )

    draw_streamline_ax(
        axes[1,0],
        pos,
        labels,
        vec,
        "Coexact Streamline",
        density=1.5,
        maxlength=10,
    )

    #################################################
    # Total vector
    #################################################

    vec = edge_flow_to_node_vector(
        edges,
        omega,
        pos,
        len(labels),
    )

    vec = smooth_node_vectors(
        edges,
        vec,
        len(labels),
        smooth=0.05,
    )

    draw_streamline_ax(
        axes[1,1],
        pos,
        labels,
        vec,
        "Total Streamline",
        density=1.5,
        maxlength=10,
    )

    plt.tight_layout()

    pdf_name = f"hodge_42_epoch/result_{index:03d}.pdf"

    with PdfPages(pdf_name) as pdf:
        pdf.savefig(fig)

    plt.close(fig)

    return pdf_name